In [ ]:
"""
System Hardening Script for Cybersecurity Research
Performs essential security configurations for dark web research
"""

import os
import subprocess
import sys
import hashlib
import logging
from datetime import datetime

# Setup secure logging
logging.basicConfig(
    filename='hardening_audit.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

def check_root():
    """Ensure script runs with root privileges"""
    if os.geteuid() != 0:
        logging.error("This script must be run as root")
        sys.exit(1)

def configure_firewall():
    """Configure basic firewall rules"""
    try:
        # Flush existing rules
        subprocess.run(['iptables', '-F'], check=True)
        subprocess.run(['iptables', '-X'], check=True)
        
        # Default deny policy
        subprocess.run(['iptables', '-P', 'INPUT', 'DROP'], check=True)
        subprocess.run(['iptables', '-P', 'FORWARD', 'DROP'], check=True)
        subprocess.run(['iptables', '-P', 'OUTPUT', 'DROP'], check=True)
        
        # Allow loopback
        subprocess.run(['iptables', '-A', 'INPUT', '-i', 'lo', '-j', 'ACCEPT'], check=True)
        subprocess.run(['iptables', '-A', 'OUTPUT', '-o', 'lo', '-j', 'ACCEPT'], check=True)
        
        # Allow established connections
        subprocess.run(['iptables', '-A', 'INPUT', '-m', 'conntrack', '--ctstate', 'ESTABLISHED,RELATED', '-j', 'ACCEPT'], check=True)
        
        # Allow outbound connections (restrict as needed)
        subprocess.run(['iptables', '-A', 'OUTPUT', '-j', 'ACCEPT'], check=True)
        
        logging.info("Firewall configured successfully")
    except subprocess.CalledProcessError as e:
        logging.error(f"Firewall configuration failed: {str(e)}")
        sys.exit(1)

def disable_unnecessary_services():
    """Disable potentially risky services"""
    services_to_disable = [
        'avahi-daemon', 'cups', 'dhcpd', 'slapd', 'nfs', 'rpcbind',
        'telnet', 'vsftpd', 'smb', 'squid', 'snmpd', 'ypserv'
    ]
    
    for service in services_to_disable:
        try:
            subprocess.run(['systemctl', 'stop', service], check=True)
            subprocess.run(['systemctl', 'disable', service], check=True)
            logging.info(f"Disabled service: {service}")
        except subprocess.CalledProcessError:
            logging.warning(f"Failed to disable service: {service} (may not be installed)")

def configure_ssh_security():
    """Harden SSH server configuration"""
    ssh_config = "/etc/ssh/sshd_config"
    backup_config = "/etc/ssh/sshd_config.bak"
    
    try:
        # Create backup
        subprocess.run(['cp', ssh_config, backup_config], check=True)
        
        # Apply security settings
        with open(ssh_config, 'a') as f:
            f.write("\n# Security Hardening Additions\n")
            f.write("PermitRootLogin no\n")
            f.write("PasswordAuthentication no\n")
            f.write("ChallengeResponseAuthentication no\n")
            f.write("X11Forwarding no\n")
            f.write("MaxAuthTries 3\n")
            f.write("ClientAliveInterval 300\n")
            f.write("ClientAliveCountMax 0\n")
            f.write("AllowUsers research_user\n")  # Change to your research account
        
        # Restart SSH
        subprocess.run(['systemctl', 'restart', 'sshd'], check=True)
        logging.info("SSH security configured")
    except Exception as e:
        logging.error(f"SSH configuration failed: {str(e)}")
        sys.exit(1)

def setup_automatic_updates():
    """Configure automatic security updates"""
    try:
        subprocess.run(['apt-get', 'install', '-y', 'unattended-upgrades'], check=True)
        
        config_file = "/etc/apt/apt.conf.d/20auto-upgrades"
        with open(config_file, 'w') as f:
            f.write('APT::Periodic::Update-Package-Lists "1";\n')
            f.write('APT::Periodic::Unattended-Upgrade "1";\n')
            f.write('APT::Periodic::AutocleanInterval "7";\n')
        
        logging.info("Automatic updates configured")
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to configure updates: {str(e)}")

def main():
    """Main hardening function"""
    logging.info("=== Starting System Hardening ===")
    check_root()
    
    print("[*] Configuring firewall...")
    configure_firewall()
    
    print("[*] Disabling unnecessary services...")
    disable_unnecessary_services()
    
    print("[*] Securing SSH...")
    configure_ssh_security()
    
    print("[*] Setting up automatic updates...")
    setup_automatic_updates()
    
    logging.info("=== System Hardening Complete ===")
    print("\n[+] System hardening completed. Review hardening_audit.log for details.")
    print("[!] Remember to configure additional security measures:")
    print("    - Full disk encryption")
    print("    - VPN configuration")
    print("    - Secure boot settings")
    print("    - BIOS/UEFI password")

if __name__ == "__main__":
    main()

Remember to 

- Run these scripts in an isolated VM (like Whonix or Tails)

- Use a VPN before connecting to Tor

- Never run as root unless absolutely necessary (system hardening requires root)

- Replace placeholder onion URLs with your actual research targets

- Adjust selectors in extract_forum_data() to match your target sites

- Modify sanitisation rules based on your data handling policies

In [ ]:
#!/usr/bin/env python3
"""
Dark Web Data Collector for Threat Intelligence Research
Ethical collection from public dark web sources with Tor
"""

import os
import time
import random
import requests
import stem.process
from stem import Signal
from stem.control import Controller
from bs4 import BeautifulSoup
import logging
from urllib.parse import urlparse
import json
from datetime import datetime

# Configuration
TOR_PORT = 9050
CONTROL_PORT = 9051
TOR_PASSWORD = "your_tor_password"  # Change this!
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; rv:91.0) Gecko/20100101 Firefox/91.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
]
MAX_PAGES = 5  # Limit for ethical collection
DELAY_RANGE = (5, 15)  # Seconds between requests
DATA_DIR = "collected_data"
LOG_FILE = "collection.log"

# Setup logging
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

class TorManager:
    """Handles Tor connection and identity rotation"""
    def __init__(self):
        self.tor_process = None
        self.controller = None
    
    def start_tor(self):
        """Start Tor process"""
        try:
            self.tor_process = stem.process.launch_tor_with_config(
                config={
                    'SocksPort': str(TOR_PORT),
                    'ControlPort': str(CONTROL_PORT),
                    'HashedControlPassword': subprocess.getoutput(f"tor --hash-password {TOR_PASSWORD}"),
                },
                init_msg_handler=self.print_bootstrap_lines
            )
            logging.info("Tor process started")
            return True
        except Exception as e:
            logging.error(f"Failed to start Tor: {str(e)}")
            return False
    
    def print_bootstrap_lines(self, line):
        """Callback for Tor bootstrap progress"""
        if "Bootstrapped" in line:
            logging.info(line)
    
    def connect_controller(self):
        """Connect to Tor control port"""
        try:
            self.controller = Controller.from_port(port=CONTROL_PORT)
            self.controller.authenticate(password=TOR_PASSWORD)
            logging.info("Connected to Tor controller")
            return True
        except Exception as e:
            logging.error(f"Controller connection failed: {str(e)}")
            return False
    
    def renew_identity(self):
        """Request new Tor identity"""
        if self.controller:
            try:
                self.controller.signal(Signal.NEWNYM)
                logging.info("Requested new Tor identity")
                # Wait for circuit to be rebuilt
                time.sleep(5)
                return True
            except Exception as e:
                logging.error(f"Failed to renew identity: {str(e)}")
                return False
        return False
    
    def stop_tor(self):
        """Stop Tor process"""
        if self.tor_process:
            self.tor_process.terminate()
            logging.info("Tor process stopped")

class DarkWebCollector:
    """Main collection class with ethical safeguards"""
    def __init__(self):
        self.tor_manager = TorManager()
        self.session = requests.Session()
        self.session.proxies = {
            'http': f'socks5h://127.0.0.1:{TOR_PORT}',
            'https': f'socks5h://127.0.0.1:{TOR_PORT}'
        }
        self.visited_links = set()
        os.makedirs(DATA_DIR, exist_ok=True)
    
    def random_delay(self):
        """Random delay between requests"""
        delay = random.randint(*DELAY_RANGE)
        time.sleep(delay)
    
    def get_user_agent(self):
        """Return random user agent"""
        return random.choice(USER_AGENTS)
    
    def sanitize_content(self, text):
        """Basic sanitization of collected content"""
        # Remove any potential PII (customize as needed)
        text = text.replace(".onion", "[REDACTED]")
        return text
    
    def is_valid_onion(self, url):
        """Validate onion URL format"""
        parsed = urlparse(url)
        return parsed.netloc.endswith('.onion') and len(parsed.netloc) == 22
    
    def fetch_page(self, url):
        """Fetch a page with ethical safeguards"""
        if not self.is_valid_onion(url):
            logging.warning(f"Invalid onion URL: {url}")
            return None
        
        if url in self.visited_links:
            logging.info(f"Already visited: {url}")
            return None
        
        self.random_delay()
        headers = {'User-Agent': self.get_user_agent()}
        
        try:
            response = self.session.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                self.visited_links.add(url)
                content = self.sanitize_content(response.text)
                return content
            else:
                logging.warning(f"HTTP {response.status_code} for {url}")
                return None
        except Exception as e:
            logging.error(f"Error fetching {url}: {str(e)}")
            return None
    
    def extract_forum_data(self, html):
        """Extract structured data from forum pages"""
        soup = BeautifulSoup(html, 'html.parser')
        data = {
            'threads': [],
            'timestamp': datetime.utcnow().isoformat()
        }
        
        # Example extraction - customize for target sites
        for thread in soup.select('.thread'):  # Update selector based on actual site
            try:
                title = thread.select_one('.title').text.strip()
                author = thread.select_one('.author').text.strip()
                date = thread.select_one('.date').text.strip()
                content = thread.select_one('.content').text.strip()
                
                data['threads'].append({
                    'title': self.sanitize_content(title),
                    'author': '[ANONYMIZED]',  # Never store actual usernames
                    'date': date,
                    'content': self.sanitize_content(content),
                    'metadata': {
                        'word_count': len(content.split()),
                        'has_attachments': bool(thread.select('.attachment'))
                    }
                })
            except Exception as e:
                logging.warning(f"Error parsing thread: {str(e)}")
                continue
        
        return data
    
    def save_data(self, data, source):
        """Save collected data with timestamp"""
        if not data or not data.get('threads'):
            return False
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{DATA_DIR}/{source}_{timestamp}.json"
        
        try:
            with open(filename, 'w') as f:
                json.dump(data, f, indent=2)
            logging.info(f"Saved data to {filename}")
            return True
        except Exception as e:
            logging.error(f"Failed to save data: {str(e)}")
            return False
    
    def collect_from_forum(self, base_url):
        """Ethical collection from a forum with pagination limits"""
        if not self.tor_manager.start_tor():
            return False
        
        if not self.tor_manager.connect_controller():
            self.tor_manager.stop_tor()
            return False
        
        try:
            current_page = 1
            while current_page <= MAX_PAGES:
                url = f"{base_url}?page={current_page}" if current_page > 1 else base_url
                logging.info(f"Collecting from: {url}")
                
                html = self.fetch_page(url)
                if not html:
                    break
                
                forum_data = self.extract_forum_data(html)
                if forum_data:
                    self.save_data(forum_data, urlparse(base_url).netloc)
                
                current_page += 1
                if current_page % 3 == 0:  # Rotate identity periodically
                    self.tor_manager.renew_identity()
            
            return True
        except Exception as e:
            logging.error(f"Collection failed: {str(e)}")
            return False
        finally:
            self.tor_manager.stop_tor()

def main():
    """Main collection routine"""
    print("=== Dark Web Data Collector ===")
    print("[!] For academic research only. Use ethically and legally.")
    
    collector = DarkWebCollector()
    
    # Example forums - that should be replaced with the actual research targets and can be modified/augmented to collect more data
    # These are example patterns (not real forums). For the script to be effective, you must identify actual research-appropriate targets.
    research_forums = [
        "http://exampleforum.onion",  # Replace with actual research-appropriate forum
        "http://anotherforum.onion",   # Replace with actual research-appropriate forum
        "http://samplesecureforum.onion"  # Replace with actual research-appropriate forum
    ]
    
    for forum in research_forums:
        print(f"\n[*] Collecting from: {forum}")
        success = collector.collect_from_forum(forum)
        status = "SUCCESS" if success else "FAILED"
        print(f"[{status}] Completed: {forum}")
    
    print("\n[+] Collection complete. Review logs in:", LOG_FILE)
    print("[!] Remember to analyze data in a secure environment")

if __name__ == "__main__":
    main()

In [ ]:
"""Script to extract dates, site names, and search for key terms in CSV and Excel files from csv data saved in a specified folder."""
import os
import re
import pandas as pd
from datetime import datetime
import glob

def extract_date_from_filename(filename):
    """
    Extract month and year from filename using common date patterns
    """
    # Common date patterns in filenames
    patterns = [
        r'(\d{1,2})[-_](\d{4})',  # MM-YYYY or MM_YYYY
        r'(\d{4})[-_](\d{1,2})',  # YYYY-MM or YYYY_MM
        r'(\w+)[-_](\d{4})',      # MonthName-YYYY or MonthName_YYYY
        r'(\d{4})',               # Just YYYY
        r'(\d{1,2})[/-](\d{1,2})[/-](\d{4})',  # MM-DD-YYYY or MM/DD/YYYY
    ]
    
    filename_lower = filename.lower()
    
    # Try MM-YYYY or MM_YYYY
    match = re.search(r'(\d{1,2})[-_](\d{4})', filename)
    if match:
        month, year = int(match.group(1)), int(match.group(2))
        return month, year
    
    # Try YYYY-MM or YYYY_MM
    match = re.search(r'(\d{4})[-_](\d{1,2})', filename)
    if match:
        year, month = int(match.group(1)), int(match.group(2))
        return month, year
    
    # Try month names
    month_names = {
        'jan': 1, 'january': 1, 'feb': 2, 'february': 2, 'mar': 3, 'march': 3,
        'apr': 4, 'april': 4, 'may': 5, 'jun': 6, 'june': 6, 'jul': 7, 'july': 7,
        'aug': 8, 'august': 8, 'sep': 9, 'september': 9, 'oct': 10, 'october': 10,
        'nov': 11, 'november': 11, 'dec': 12, 'december': 12
    }
    
    for month_name, month_num in month_names.items():
        pattern = f'{month_name}[-_](\d{{4}})'
        match = re.search(pattern, filename_lower)
        if match:
            year = int(match.group(1))
            return month_num, year
    
    # Try just year
    match = re.search(r'(\d{4})', filename)
    if match:
        year = int(match.group(1))
        return None, year  # Month unknown
    
    return None, None

def extract_site_from_filename(filename):
    """
    Extract site name from filename
    """
    # Remove file extension
    name_without_ext = os.path.splitext(filename)[0]
    
    # Common patterns for site extraction
    # Look for patterns like "site_something", "something_site", etc.
    patterns = [
        r'([a-zA-Z0-9]+)[-_]',  # text before separator
        r'[-_]([a-zA-Z0-9]+)',  # text after separator
    ]
    
    # Try to extract site name by splitting on common separators
    separators = ['_', '-', ' ']
    for sep in separators:
        parts = name_without_ext.split(sep)
        if len(parts) > 1:
            # Return the part that looks most like a site name (not date, not too short)
            for part in parts:
                if (len(part) >= 3 and 
                    not part.isdigit() and 
                    not re.match(r'^\d{1,2}$', part) and
                    not re.match(r'^\d{4}$', part)):
                    return part
    
    return name_without_ext  # Return filename without extension as fallback

def search_key_terms(text, search_list, mention_list):
    """
    Search for key terms in text and return matched terms
    """
    matched_terms = []
    text_lower = str(text).lower()
    
    # Search through search_list patterns
    for search_pattern in search_list:
        # Extract individual terms from the OR patterns
        terms = re.findall(r'"([^"]*)"', search_pattern)
        for term in terms:
            if term.lower() in text_lower:
                matched_terms.append(term)
    
    # Search through mention_list tuples
    for term_group in mention_list:
        for term in term_group:
            if term.lower() in text_lower:
                matched_terms.append(term)
    
    return list(set(matched_terms))  # Remove duplicates

def process_file(file_path):
    """
    Process a single CSV or Excel file
    """
    filename = os.path.basename(file_path)
    month, year = extract_date_from_filename(filename)
    site = extract_site_from_filename(filename)
    
    print(f"Processing: {filename}")
    print(f"  Date: Month={month}, Year={year}")
    print(f"  Site: {site}")
    
    try:
        # Read file based on extension
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        elif file_path.endswith(('.xls', '.xlsx')):
            # Read all sheets if Excel file
            xl_file = pd.ExcelFile(file_path)
            results = []
            
            for sheet_name in xl_file.sheet_names:
                df = xl_file.parse(sheet_name)
                sheet_results = process_dataframe(df, filename, month, year, site, sheet_name)
                results.extend(sheet_results)
            
            return results
        else:
            return []
        
        return process_dataframe(df, filename, month, year, site)
        
    except Exception as e:
        print(f"  Error processing file {filename}: {str(e)}")
        return []

def process_dataframe(df, filename, month, year, site, sheet_name=None):
    """
    Process a DataFrame and search for key terms
    """
    results = []
    
    # Check if 'price' column exists
    if 'price' not in df.columns:
        print(f"  Warning: 'price' column not found in {filename}")
        # Try to find price column with case-insensitive search
        price_cols = [col for col in df.columns if 'price' in col.lower()]
        if price_cols:
            price_col = price_cols[0]
            print(f"  Using alternative price column: {price_col}")
        else:
            return results
    else:
        price_col = 'price'
    
    # Iterate through each row
    for idx, row in df.iterrows():
        # Combine all text columns for searching
        text_to_search = ' '.join([str(val) for val in row.values if pd.notna(val)])
        
        # Search for key terms
        matched_terms = search_key_terms(text_to_search, search_list, mention_list)
        
        if matched_terms:
            price = row[price_col] if pd.notna(row[price_col]) else None
            
            result = {
                'filename': filename,
                'sheet_name': sheet_name,
                'row_index': idx,
                'month': month,
                'year': year,
                'site': site,
                'matched_terms': ', '.join(matched_terms),
                'price': price,
                'all_data': str(row.to_dict())
            }
            results.append(result)
            
            print(f"    Row {idx}: Found terms: {matched_terms}, Price: {price}")
    
    return results

def main():
    # Define your folder path here
    folder_path = input("Enter the folder path containing CSV and Excel files: ").strip()
    
    if not os.path.exists(folder_path):
        print("Folder path does not exist!")
        return
    
    # Find all CSV and Excel files
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    excel_files = glob.glob(os.path.join(folder_path, "*.xls")) + glob.glob(os.path.join(folder_path, "*.xlsx"))
    all_files = csv_files + excel_files
    
    if not all_files:
        print("No CSV or Excel files found in the specified folder!")
        return
    
    print(f"Found {len(all_files)} files to process")
    print("=" * 50)
    
    all_results = []
    
    # Process each file
    for file_path in all_files:
        file_results = process_file(file_path)
        all_results.extend(file_results)
        print("-" * 30)
    
    # Create final DataFrame
    if all_results:
        results_df = pd.DataFrame(all_results)
        
        # Save results to CSV
        output_file = "scraping_results.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\nResults saved to {output_file}")
        print(f"Total matches found: {len(all_results)}")
        
        # Display summary
        print("\nSummary:")
        print(f"Files processed: {len(all_files)}")
        print(f"Total matches: {len(all_results)}")
        print(f"Unique terms found: {len(set([term for result in all_results for term in result['matched_terms'].split(', ')]))}")
        
    else:
        print("No key terms found in any files.")

if __name__ == "__main__":
    # Define the search lists (copy your lists here)
    search_list = [
        # --- INITIAL ACCESS & SOCIAL ENGINEERING ---
        # Phishing and Social Engineering
        '"Phishing" OR "Spear Phishing" OR "Whaling" OR "Business Email Compromise" OR "BEC" OR "Email spoofing"',
        '"Social engineering" OR "Pretexting" OR "Baiting" OR "Quid pro quo" OR "Vishing" OR "Smishing"',
        
        # Email-Based Attacks
        '"Email bombing" OR "Email flood" OR "Mailbomb" OR "Spam attack" OR "Malicious email attachment" OR "Malicious link"',
        
        # Drive-by & Watering Hole Attacks
        '"Drive-by download" OR "Drive-by install" OR "Drive-by attack" OR "Malicious redirect" OR "Watering hole attack"',
        
        # Exploit Kits & Vulnerability Sales
        '"Exploit kit" OR "EK" OR "Angler" OR "Neutrino" OR "Blackhole" OR "Rig" OR "Sundown"',
        '"Zero-day" OR "0-day" OR "Zero day exploit" OR ("vulnerability" AND (exploit OR sale))',
        
        # --- MALWARE & DELIVERY ---
        # Malware (Umbrella)
        '"Malware" OR "Malicious software" OR "Malicious code" OR "Malicious payload"',
        
        # Ransomware & Extortion
        '"Ransomware" OR "Crypto-ransomware" OR "Locker ransomware" OR "Double extortion" OR "Doxware"',
        # Ransomware-as-a-Service (RaaS)
        '"Ransomware as a Service" OR "RaaS" OR "RaaS affiliate"',
        # Specific Ransomware Variants
        '"WannaCry" OR "WannaCrypt" OR "Petya" OR "NotPetya" OR "Cryptolocker" OR "Locky" OR "REvil" OR "Sodinokibi" OR "Maze" OR "LockBit" OR "DarkSide" OR "BlackMatter" OR "Conti" OR "BlackCat" OR "ALPHV"',
        
        # Trojans & Backdoors
        '"Trojan" OR "Trojan horse" OR "Banking trojan" OR "Dropper" OR "Downloader"',
        '"Backdoor" OR "Remote Access Trojan" OR "RAT" OR "Web shell" OR ("C99" AND "shell")',
        
        # Info-Stealers & Spyware
        '"Info-stealer" OR "Infostealer" OR "Information stealer" OR "Spyware" OR "Keylogger" OR "Keystroke logger"',
        '"Pegasus" OR "Predator" OR "FlexiSpy" OR "Redline Stealer" OR "Vidar" OR "Racoon Stealer"',
        
        # Worms & Viruses
        '"Computer worm" OR "Worm" OR "Self-propagating malware" OR "Network worm" OR "Computer virus" OR "Virus"',
        
        # Botnets & Command & Control
        '"Botnet" OR "Bot herder" OR "C2 server" OR "Command and control" OR "Zombie army" OR "DDoS bot"',
        
        # Rootkits & Bootkits
        '"Rootkit" OR "Bootkit" OR "Kernel rootkit" OR "Hypervisor rootkit"',
        
        # Adware & Unwanted Software
        '"Adware" OR "Malvertising" OR "Malicious advertising" OR "PUPs" OR "Potentially Unwanted Program"',
        
        # Wiper Malware
        '"Wiper malware" OR "Wiper" OR "Data destruction malware" OR "HermeticWiper" OR "IsaacWiper" OR "Shamoon"',
        
        # Malware-as-a-Service (MaaS)
        '"Malware as a Service" OR "MaaS" OR "Crimeware as a Service" OR "CaaS"',
        
        # --- NETWORK & APPLICATION ATTACKS ---
        # Denial of Service Attacks
        '"Denial-of-Service" OR "DoS" OR "DDoS" OR "Distributed Denial-of-Service" OR "HTTP flood" OR "SYN flood" OR "Amplification attack"',
        
        # Web Application Attacks
        '"SQL Injection" OR "SQLi" OR "SQLI" OR "Cross-site scripting" OR "XSS" OR "Stored XSS" OR "Reflected XSS"',
        '"Remote Code Execution" OR "RCE" OR "Local File Inclusion" OR "LFI" OR "Remote File Inclusion" OR "RFI"',
        
        # Man-in-the-Middle & Session Attacks
        '"Man-in-the-middle" OR "MITM" OR "MitM" OR "Session hijacking" OR "Cookie hijacking" OR "Cookie stealing"',
        
        # DNS & Pharming Attacks
        '"DNS spoofing" OR "DNS cache poisoning" OR "DNS hijacking" OR "DNS tunneling" OR "Pharming"',
        
        # Password & Authentication Attacks
        '"Password attack" OR "Credential theft" OR "Credential stuffing" OR "Brute force" OR "Dictionary attack" OR "Password spraying" OR "Pass the hash"',
        
        # --- MONETIZATION & FRAUD (DARK WEB FOCUS) ---
        # Carding & Financial Fraud
        '"Carding" OR "Card fraud" OR "Credit card theft" OR "Payment card fraud" OR "Fullz" OR "Fulls" OR "Card dump" OR "Dump" OR "BIN" OR "CVV" OR "CVV2"',
        '"Bank log" OR "Account takeover" OR "ATO" OR "eWhoring"',
        
        # Cryptojacking
        '"Cryptojacking" OR "Illegal cryptocurrency mining" OR "Coin miner malware" OR "Crypto-miner"',
        
        # Data Breaches & Leaks
        '"Data breach" OR "Data leak" OR "Data exfiltration" OR "Data theft" OR "Database dump" OR "Leaked database"',
        
        # Underground Markets & Services
        '"Dark web market" OR "Underground market" OR "Darknet market" OR "Carding forum" OR "Hacking forum"',
        '"Doxing" OR "Doxxing" OR "Swatting"',
        
        # --- PERSISTENCE, EVASION & ADVANCED THREATS ---
        # Lateral Movement & Persistence
        '"Lateral movement" OR "Privilege escalation" OR "Persistence" OR "Defense evasion" OR "Living off the land" OR "LotL"',
        
        # Advanced Persistent Threats (APTs)
        '"Advanced Persistent Threat" OR "APT" OR "Nation state attack" OR "State-sponsored"',
        
        # Supply Chain Attacks
        '"Supply chain attack" OR "Third-party attack" OR "Software supply chain" OR "Dependency confusion"',
        
        # Insider Threats
        '"Insider threat" OR "Malicious insider" OR "Rogue employee" OR "Insider fraud"',
        
        # IoT & Operational Technology Attacks
        '"IoT attack" OR "IoT botnet" OR "Mirai" OR "Operational Technology" OR "OT attack" OR "ICS attack"',
        
        # AI Security & Adversarial Attacks
        '"Adversarial attack" OR "AI poisoning" OR "Model inversion" OR "Data poisoning" OR "Training data poisoning"',
    ]

    mention_list = [
        # Initial Access
        ('Phishing', 'Spear Phishing', 'Whaling', 'BEC', 'Email Spoofing'),
        ('Social Engineering', 'Pretexting', 'Baiting', 'Vishing', 'Smishing'),
        ('Email Bombing', 'Mailbomb', 'Spam Attack'),
        ('Drive-by Download', 'Watering Hole Attack'),
        ('Exploit Kit', 'Zero-day', '0-day'),
        # Malware
        ('Malware', 'Malicious Software'),
        ('Ransomware', 'Crypto-ransomware', 'Locker Ransomware'),
        ('Ransomware-as-a-Service', 'RaaS'),
        ('Trojan', 'Banking Trojan', 'Dropper', 'Downloader'),
        ('Backdoor', 'RAT', 'Web Shell'),
        ('Info-Stealer', 'Spyware', 'Keylogger'),
        ('Worm', 'Virus', 'Self-propagating Malware'),
        ('Botnet', 'C2 Server', 'Command and Control'),
        ('Rootkit', 'Bootkit'),
        ('Adware', 'Malvertising', 'PUPs'),
        ('Wiper Malware', 'Data Destruction Malware'),
        ('Malware-as-a-Service', 'MaaS', 'CaaS'),
        # Network & App Attacks
        ('Denial-of-Service', 'DoS', 'DDoS'),
        ('SQL Injection', 'SQLi'),
        ('Cross-site Scripting', 'XSS'),
        ('Remote Code Execution', 'RCE'),
        ('Man-in-the-Middle', 'MITM', 'Session Hijacking'),
        ('DNS Spoofing', 'DNS Hijacking', 'Pharming'),
        ('Password Attack', 'Credential Stuffing', 'Brute Force'),
        # Monetization & Fraud
        ('Carding', 'Card Fraud', 'Fullz', 'Card Dump', 'CVV'),
        ('Account Takeover', 'ATO', 'Bank Log'),
        ('Cryptojacking', 'Illegal Cryptomining'),
        ('Data Breach', 'Data Leak', 'Data Exfiltration'),
        ('Dark Web Market', 'Underground Market', 'Carding Forum'),
        # Advanced Tactics
        ('Lateral Movement', 'Privilege Escalation', 'Persistence'),
        ('Advanced Persistent Threat', 'APT'),
        ('Supply Chain Attack', 'Third-party Attack'),
        ('Insider Threat', 'Malicious Insider'),
        ('IoT Attack', 'OT Attack', 'ICS Attack'),
        ('Adversarial Attack', 'AI Poisoning', 'Data Poisoning'),
    ]

    main()

1. Place all the csv files from forums scraped in a folder
2. Run the script to return mention counts

In [ ]:
"""Data cleaning and transformation script for market data from 2020 to 2024 to find unique terms and an average clean price for each term mentioned."""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

def clean_price(price_str):
    """Extract numeric value from price string"""
    if pd.isna(price_str):
        return np.nan
    
    # Remove currency symbols and convert to float
    price_str = str(price_str).upper()
    # Extract numbers with optional decimal point
    match = re.search(r'(\d+\.?\d*)', price_str)
    if match:
        return float(match.group(1))
    return np.nan

def normalize_term(term):
    """Normalise terms by converting to lowercase and stripping whitespace"""
    return term.strip().lower()

def get_unique_terms(matched_terms_str):
    """Extract unique terms from matched_terms string, handling duplicates and variations"""
    if pd.isna(matched_terms_str):
        return set()
    
    terms = str(matched_terms_str).split(',')
    unique_terms = set()
    
    for term in terms:
        normalized = normalize_term(term)
        if normalized:  # Only add non-empty terms
            unique_terms.add(normalized)
    
    return unique_terms

def generate_complete_date_range():
    """Generate complete date range from Dec 2020 to Oct 2024"""
    dates = []
    start_date = datetime(2020, 12, 1)
    end_date = datetime(2024, 10, 1)
    
    current_date = start_date
    while current_date <= end_date:
        dates.append(f"{current_date.strftime('%B')}-{current_date.year}")
        # Move to next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)
    
    return dates

def main():
    # Read the input CSV file
    input_file = "market data 2020-2024.csv" # Change this to your actual file path
    
    try:
        df = pd.read_csv(input_file)
        print(f"Successfully loaded data with {len(df)} rows")
        print(f"Columns: {df.columns.tolist()}")
        print(f"Data types:\n{df.dtypes}")
    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found. Please check the file path.")
        return
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    
    # Check for required columns
    required_columns = ['month', 'year', 'matched_terms', 'price']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"Error: Missing required columns: {missing_columns}")
        return
    
    # Clean and convert month to integer
    df['month'] = df['month'].fillna(0).astype(int)
    
    # Filter out rows with invalid month (0 or out of range)
    df = df[(df['month'] >= 1) & (df['month'] <= 12)]
    
    print(f"Rows after month cleaning: {len(df)}")
    
    # Clean the price column
    df['price_clean'] = df['price'].apply(clean_price)
    
    # Define all the terms we're looking for (as per previous lists)
    all_terms = {
        'carding', 'cvv', 'fullz', 'rat', 'malware', 'ato', 'ek', 'rig', 'bin', 'phishing',
        'dump', 'apt', 'rce', 'bec', 'bank log', 'lfi', 'botnet', 'keylogger', 'virus',
        'pegasus', 'ransomware', 'sql injection', 'trojan', 'vulnerability', 'ewhoring',
        'rfi', 'shell', 'doxing', 'downloader', 'card fraud', 'dark web market', 'redline stealer',
        'ddos', 'exploit kit', 'conti', 'maze', 'predator', 'xss', 'worm', 'cryptolocker',
        'blackcat', 'leaked database', 'spyware', 'brute force', 'hacking forum', 'dictionary attack',
        'angler', 'carding forum', 'neutrino', 'persistence', 'revil', 'zero-day', 'mirai',
        'credential stuffing', 'bootkit', 'c2 server', 'darkside', 'data leak', 'data breach',
        'email flood', 'email spoofing', 'privilege escalation', 'cookie stealing', 'insider fraud',
        'adware', 'dropper', 'remote code execution', 'sql', 'sqli', 'social engineering'
    }
    
    # Create a mapping from normalised terms to their display names
    term_display_names = {
        'carding': 'Carding',
        'cvv': 'CVV', 
        'fullz': 'Fullz',
        'rat': 'RAT',
        'malware': 'Malware',
        'ato': 'ATO',
        'ek': 'EK',
        'rig': 'Rig',
        'bin': 'BIN',
        'phishing': 'Phishing',
        'dump': 'Dump',
        'apt': 'APT',
        'rce': 'RCE',
        'bec': 'BEC',
        'bank log': 'Bank Log',
        'lfi': 'LFI',
        'botnet': 'Botnet',
        'keylogger': 'Keylogger',
        'virus': 'Virus',
        'pegasus': 'Pegasus',
        'ransomware': 'Ransomware',
        'sql injection': 'SQL Injection',
        'trojan': 'Trojan',
        'vulnerability': 'Vulnerability',
        'ewhoring': 'eWhoring',
        'rfi': 'RFI',
        'shell': 'Shell',
        'doxing': 'Doxing',
        'downloader': 'Downloader',
        'card fraud': 'Card Fraud',
        'dark web market': 'Dark Web Market',
        'redline stealer': 'Redline Stealer',
        'ddos': 'DDoS',
        'exploit kit': 'Exploit Kit',
        'conti': 'Conti',
        'maze': 'Maze',
        'predator': 'Predator',
        'xss': 'XSS',
        'worm': 'Worm',
        'cryptolocker': 'Cryptolocker',
        'blackcat': 'BlackCat',
        'leaked database': 'Leaked Database',
        'spyware': 'Spyware',
        'brute force': 'Brute Force',
        'hacking forum': 'Hacking Forum',
        'dictionary attack': 'Dictionary Attack',
        'angler': 'Angler',
        'carding forum': 'Carding Forum',
        'neutrino': 'Neutrino',
        'persistence': 'Persistence',
        'revil': 'REvil',
        'zero-day': 'Zero-day',
        'mirai': 'Mirai',
        'credential stuffing': 'Credential Stuffing',
        'bootkit': 'Bootkit',
        'c2 server': 'C2 Server',
        'darkside': 'DarkSide',
        'data leak': 'Data Leak',
        'data breach': 'Data Breach',
        'email flood': 'Email Flood',
        'email spoofing': 'Email Spoofing',
        'privilege escalation': 'Privilege Escalation',
        'cookie stealing': 'Cookie Stealing',
        'insider fraud': 'Insider Fraud',
        'adware': 'Adware',
        'dropper': 'Dropper',
        'remote code execution': 'Remote Code Execution',
        'sql': 'SQL',
        'sqli': 'SQLi',
        'social engineering': 'Social Engineering'
    }
    
    # Generate complete date range
    complete_dates = generate_complete_date_range()
    print(f"Generated date range: {len(complete_dates)} months from {complete_dates[0]} to {complete_dates[-1]}")
    
    # Prepare data for monthly counts - start with complete date range
    monthly_data = []
    
    # Initialise all dates with zeros
    for date_str in complete_dates:
        row_data = {'Date': date_str}
        for term in all_terms:
            display_name = term_display_names.get(term, term.title())
            row_data[display_name] = 0
        monthly_data.append(row_data)
    
    # Process each row to extract unique terms and update counts
    for idx, row in df.iterrows():
        month = row['month']
        year = row['year']
        matched_terms = row['matched_terms']
        
        # Get unique terms for this row
        unique_terms = get_unique_terms(matched_terms)
        
        # Create date string in format "Month-Year"
        try:
            date_str = f"{datetime(2000, int(month), 1).strftime('%B')}-{int(year)}"
        except (ValueError, TypeError) as e:
            print(f"Skipping row {idx} with invalid date: month={month}, year={year}")
            continue
        
        # Update counts for this date
        for data in monthly_data:
            if data['Date'] == date_str:
                for term in all_terms:
                    if term in unique_terms:
                        display_name = term_display_names.get(term, term.title())
                        data[display_name] += 1
                break
    
    # Convert to DataFrame for monthly counts
    monthly_counts_df = pd.DataFrame(monthly_data)
    
    # Sort by year and month
    def sort_key(date_str):
        try:
            month_str, year_str = date_str.split('-')
            month_num = datetime.strptime(month_str, '%B').month
            return (int(year_str), month_num)
        except:
            return (9999, 13)  # Put invalid dates at the end
    
    monthly_counts_df['sort_key'] = monthly_counts_df['Date'].apply(sort_key)
    monthly_counts_df = monthly_counts_df.sort_values('sort_key').drop('sort_key', axis=1)
    
    # Save monthly counts to CSV
    monthly_counts_df.to_csv('monthly_term_counts.csv', index=False)
    print("Monthly term counts saved to 'monthly_term_counts.csv'")
    
    # Prepare data for average prices
    price_data = []
    
    # Process each row for average prices
    for idx, row in df.iterrows():
        month = row['month']
        year = row['year']
        price = row['price_clean']
        matched_terms = row['matched_terms']
        
        if pd.isna(price):
            continue
        
        # Get unique terms for this row
        unique_terms = get_unique_terms(matched_terms)
        
        # Create date string
        try:
            date_str = f"{datetime(2000, int(month), 1).strftime('%B')}-{int(year)}"
        except (ValueError, TypeError):
            continue
        
        # Add entry for each term found
        for term in unique_terms:
            if term in all_terms:
                display_name = term_display_names.get(term, term.title())
                price_data.append({
                    'Date': date_str,
                    'Term': display_name,
                    'Price': price
                })
    
    
    # First, create a complete dataframe with all dates and terms
    avg_prices_complete_data = []
    
    for date_str in complete_dates:
        row_data = {'Date': date_str}
        for term in all_terms:
            display_name = term_display_names.get(term, term.title())
            row_data[display_name] = 0.0
        avg_prices_complete_data.append(row_data)
    
    avg_prices_df_complete = pd.DataFrame(avg_prices_complete_data)
    
    if price_data:
        price_df = pd.DataFrame(price_data)
        
        # Calculate average prices per term per month
        avg_prices_df = price_df.groupby(['Date', 'Term'])['Price'].mean().reset_index()
        
        # Update the complete dataframe with actual averages
        for _, row in avg_prices_df.iterrows():
            date_str = row['Date']
            term = row['Term']
            avg_price = row['Price']
            
            # Update the corresponding cell in the complete dataframe
            mask = avg_prices_df_complete['Date'] == date_str
            if term in avg_prices_df_complete.columns:
                avg_prices_df_complete.loc[mask, term] = avg_price
    
    # Sort by date
    avg_prices_df_complete['sort_key'] = avg_prices_df_complete['Date'].apply(sort_key)
    avg_prices_df_complete = avg_prices_df_complete.sort_values('sort_key').drop('sort_key', axis=1)
    
    # Save average prices to CSV
    avg_prices_df_complete.to_csv('monthly_average_prices.csv', index=False)
    print("Monthly average prices saved to 'monthly_average_prices.csv'")
    
    # Print summary statistics
    print(f"\nSummary:")
    print(f"Total rows processed: {len(df)}")
    print(f"Time period covered: {monthly_counts_df['Date'].min()} to {monthly_counts_df['Date'].max()}")
    print(f"Total months in output: {len(complete_dates)}")
    print(f"Unique terms tracked: {len(all_terms)}")
    print(f"Output files generated:")
    print(f"  - monthly_term_counts.csv: Monthly counts for each term (all months Dec 2020 - Oct 2024)")
    print(f"  - monthly_average_prices.csv: Monthly average prices for each term (all months Dec 2020 - Oct 2024)")

if __name__ == "__main__":
    main()